[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_05_confidentiality/Lab_5b_Model_Inversion_Attacks_and_Defenses.ipynb)

# **Lab 5b: Model Inversion Attacks & Privacy Defenses**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 5:** Confidentiality & Privacy Auditing


## **Learning Objectives**

By the end of this lab, you will be able to:

- **Formulate Inversion Threat Models:** Understand how gradient feedback and high-confidence predictions permit reconstruction of sensitive training features.
- **Implement Gradient-Based Inversion:** Build an end-to-end optimization pipeline using gradient ascent, Total Variation (TV) smoothing, and $L_2$ regularization to reconstruct facial biometric archetypes.
- **Quantify Biometric Leakage:** Compare reconstructed representations against ground-truth training centroids to measure perceptual and structural fidelity.
- **Implement Privacy Defenses:** Deploy output sanitization techniques including confidence rounding, argmax hard-label truncation, and logit noise injection (differential privacy mechanism).
- **Evaluate the Utility-Privacy Trade-off:** Measure how defense mechanisms degrade attack reconstruction fidelity while quantifying impacts on classification accuracy.


## **Table of Contents**

1. [Threat Model: Biometric Data Reconstruction](#threat-model)
2. [Model Inversion Theory & Total Variation Regularization](#theory)
3. [Setup & Dataset Preparation (Olivetti Faces)](#setup)
4. [FaceNet Architecture & Baseline Model Training](#models)
5. [White-Box Gradient Ascent Model Inversion](#inversion)
6. [Defenses Against Model Inversion](#defenses)
7. [Comparative Reconstruction & Defense Evaluation](#evaluation)
8. [Hands-on Exploratory Exercises](#exercises)
9. [Conclusion & Next Steps](#conclusion)


## **1. Threat Model: Biometric Data Reconstruction** <a name="threat-model"></a>

In a **Model Inversion Attack**, an adversary attempts to reconstruct recognizable representatives of the private training data by querying the trained model or exploiting access to its parameters.

Formally demonstrated by [Fredrikson et al. (CCS 2015)](https://doi.org/10.1145/2810103.2813677), model inversion exploits the fact that deep neural networks encode rich structural information about their training classes in their decision manifolds.

### **Real-World Scenarios**

| Application | Training Data | Attacker Goal | Privacy & Security Impact |
|:---|:---|:---|:---|
| **Facial Recognition** | Biometric mugshots / employee IDs | Reconstruct recognizable face of a specific subject ID | Identity theft, spoofing presentation attacks |
| **Medical Imaging** | Diagnostic MRI / CT scans | Recover pathological lesion archetypes | Medical history leakage, HIPAA / GDPR violation |
| **Pharmacogenomics** | Patient genetic markers & drug dosages | Infer sensitive genetic markers given dosage prediction | Unlawful genetic discrimination |
| **Voice Verification** | User voice acoustic embeddings | Synthesize voice spectrograms matching target identity | Deepfake audio authentication bypass |

### **Threat Assumptions**

- **Target Knowledge:** The adversary targets a specific class label $y \in \{0, \dots, C-1\}$ (e.g. "Subject 3" in a face authentication system).
- **Access Level:** 
  - *White-Box:* The adversary has access to model parameters $\theta$ and can backpropagate loss gradients $\nabla_{\mathbf{x}} \mathcal{L}(\mathbf{x}, y)$ directly with respect to input $\mathbf{x}$.
  - *Black-Box:* The adversary queries the model and obtains continuous prediction probabilities $f_y(\mathbf{x})$, estimating gradients numerically via finite differences.


## **2. Model Inversion Theory & Total Variation Regularization** <a name="theory"></a>

Model inversion frames reconstruction as a continuous optimization problem over input space $\mathbf{x} \in \mathbb{R}^{d}$:

$$\mathbf{x}^* = \arg\min_{\mathbf{x}} \left[ -\log f_y(\mathbf{x}) + \lambda_{\text{TV}} \mathcal{R}_{\text{TV}}(\mathbf{x}) + \lambda_2 \|\mathbf{x}\|_2^2 \right]$$

### **Why Natural Image Priors Are Essential**

Unconstrained gradient ascent produces noisy, high-frequency adversarial patterns that maximize class confidence without resembling natural human faces. To guide the optimization toward natural biometric features, we impose two image priors:

1. **Total Variation (TV) Regularization:** Penalizes sharp pixel differences between adjacent pixels, enforcing spatial smoothness and eliminating salt-and-pepper noise:
   $$\mathcal{R}_{\text{TV}}(\mathbf{x}) = \sum_{i, j} \left( (\mathbf{x}_{i+1, j} - \mathbf{x}_{i, j})^2 + (\mathbf{x}_{i, j+1} - \mathbf{x}_{i, j})^2 \right)$$

2. **$L_2$ Energy Penalty:** Prevents extreme pixel saturation and constrains values within the valid intensity bounds $[0, 1]$:
   $$\mathcal{R}_{2}(\mathbf{x}) = \|\mathbf{x}\|_2^2$$


## **3. Setup & Dataset Preparation (Olivetti Faces)** <a name="setup"></a>

We utilize the **AT&T Olivetti Faces** dataset: 400 grayscale images ($64 \times 64$ pixels = 4,096 features) covering 40 distinct subjects under varying lighting, facial expressions, and angles.

We select the first 10 subjects ($N=100$ images total) and partition them into training ($70\%$) and validation ($30\%$) subsets.


In [ ]:
# Install dependencies for Google Colab
%pip install -q scikit-learn matplotlib torch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import torch.nn.functional as F
from sklearn.datasets import load_breast_cancer, fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X_faces = faces.data
y_faces = faces.target

N_SUBJECTS = 10
subject_mask = y_faces < N_SUBJECTS
X_faces = X_faces[subject_mask]
y_faces = y_faces[subject_mask]

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_faces, y_faces, test_size=0.2, random_state=42, stratify=y_faces
)

## **4. FaceNet Architecture & Baseline Model Training** <a name="models"></a>

We construct `FaceNet`, a multi-layer perceptron (MLP) classification network:
- Input: Flattened $64 \times 64 = 4,096$-dimensional vector
- Hidden Layer 1: $4,096 \to 512$ with ReLU and Dropout ($p=0.30$)
- Hidden Layer 2: $512 \to 128$ with ReLU
- Output: $128 \to 10$ subject logits


In [ ]:
class FaceNet(nn.Module):
    def __init__(self, in_dim=4096, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
Xt = torch.tensor(Xf_tr, dtype=torch.float32, device=device)
yt = torch.tensor(yf_tr, dtype=torch.long, device=device)
Xv = torch.tensor(Xf_te, dtype=torch.float32, device=device)
yv = torch.tensor(yf_te, dtype=torch.long, device=device)

face_model = FaceNet(n_classes=N_SUBJECTS).to(device)
face_optimizer = optim.Adam(face_model.parameters(), lr=1e-3, weight_decay=1e-4)
face_criterion = nn.CrossEntropyLoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    face_model.train()
    face_optimizer.zero_grad()
    face_loss = face_criterion(face_model(Xt), yt)
    face_loss.backward()
    face_optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch:2d}/{EPOCHS} - loss: {face_loss.item():.4f}')

face_model.eval()
with torch.no_grad():
    tr_acc = (face_model(Xt).argmax(1) == yt).float().mean().item()
    te_acc = (face_model(Xv).argmax(1) == yv).float().mean().item()
print(f'Face model | Train acc: {tr_acc:.3f}, Test acc: {te_acc:.3f}')

## **5. White-Box Gradient Ascent Model Inversion** <a name="inversion"></a>

Starting from pure uniform noise $\mathbf{x}_0 \sim \mathcal{U}(0, 1)$, we perform gradient ascent to maximize the target subject's logit $z_{target}(\mathbf{x})$ while penalizing Total Variation and $L_2$ norm.


In [ ]:
def invert_image(model, target_class, n_iter=2000, lr=0.01, tv_weight=1e-4, l2_weight=1e-3):
    model.eval()

    # tensor of random noise
    x = torch.rand(1, 4096, requires_grad=True, device=device)
    opt = optim.Adam([x], lr=lr)
    
    for i in range(n_iter):
        opt.zero_grad()
        logits = model(x)
        loss_cls = -logits[0, target_class]
        loss_l2 = l2_weight * (x ** 2).sum()
        img = x.view(1, 64, 64)
        tv = ((img[:, 1:, :] - img[:, :-1, :]) ** 2).sum() + ((img[:, :, 1:] - img[:, :, :-1]) ** 2).sum()
        loss_tv = tv_weight * tv
        loss = loss_cls + loss_l2 + loss_tv
        loss.backward()
        opt.step()
        with torch.no_grad():
            x.clamp_(0, 1)
        if (i + 1) % 500 == 0:
            print(f'  Iteration {i + 1}/{n_iter}: confidence={torch.softmax(logits, dim=1)[0, target_class].item():.4f}')
    return x.detach().cpu().numpy().reshape(64, 64)

In [ ]:
TARGET_SUBJECTS = [0, 1, 2, 3]
inverted_faces = {}

print(f'Starting Model Inversion (Gradient Ascent) for {len(TARGET_SUBJECTS)} subjects...')

for subject_id in TARGET_SUBJECTS:
    print(f'\n>>> Inverting Subject ID: {subject_id}')
    inverted_faces[subject_id] = invert_image(face_model, subject_id)

In [ ]:
fig, axes = plt.subplots(2, len(TARGET_SUBJECTS), figsize=(3 * len(TARGET_SUBJECTS), 6))

for col, subj in enumerate(TARGET_SUBJECTS):
    true_mean_face = Xf_tr[yf_tr == subj].mean(axis=0).reshape(64, 64)
    axes[0, col].imshow(true_mean_face, cmap='gray')
    axes[0, col].set_title(f'Subject {subj}\nTrue mean')
    axes[0, col].axis('off')

    axes[1, col].imshow(inverted_faces[subj], cmap='gray')
    axes[1, col].set_title(f'Subject {subj}\nInverted')
    axes[1, col].axis('off')

plt.suptitle('Model Inversion: True Class Mean vs Gradient-Ascent Reconstruction', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## **6. Defenses Against Model Inversion** <a name="defenses"></a>

How do we prevent adversaries from reconstructing facial biometrics from trained models? Several defense strategies exist at different stages of the ML lifecycle:

### **Defense Strategies**

1. **Confidence Truncation & Rounding (Output Sanitization):**
   Exposing raw floating-point probabilities (e.g. `0.9841285...`) grants attackers fine-grained gradient signal. By rounding probabilities to 1 decimal place or suppressing non-top classes, the gradient signal is substantially degraded.
2. **Argmax (Hard-Label Only):**
   Returning only the discrete predicted identity ($y^* = \arg\max_c f_c(\mathbf{x})$) completely eliminates continuous confidence gradients $\nabla_{\mathbf{x}} f_y(\mathbf{x})$.
3. **Logit Noise Injection (Differential Privacy / Purifier):**
   Adding calibrated noise $\mathbf{\epsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I})$ or Laplace noise to logits before computing softmax masks the internal confidence differences.
4. **Gradient Obfuscation / Defensive Regularization during Training:**
   Penalizing input gradients $\|\nabla_{\mathbf{x}} \mathcal{L}\|$ during training or training with DP-SGD (Differential Privacy).

Below, we implement an output-sanitization defense wrapper: `DefendedFaceClassifier` with configurable logit perturbation and confidence rounding.


In [ ]:
class DefendedFaceClassifier(nn.Module):
    """Defense wrapper applying logit noise and output sanitization against model inversion."""
    
    def __init__(self, base_model: nn.Module, noise_std: float = 1.5, round_decimals: int = 1):
        super().__init__()
        self.base_model = base_model
        self.noise_std = noise_std
        self.round_decimals = round_decimals
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.base_model(x)
        if self.training:
            return logits
        
        # Inject calibrated Gaussian noise into logits (Differential Privacy mechanism)
        if self.noise_std > 0:
            noise = torch.randn_like(logits) * self.noise_std
            logits = logits + noise
            
        return logits
    
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Sanitized softmax probabilities with rounding."""
        with torch.no_grad():
            logits = self.forward(x)
            probs = torch.softmax(logits, dim=1)
            if self.round_decimals is not None:
                # Round probabilities to coarse precision
                factor = 10 ** self.round_decimals
                probs = torch.round(probs * factor) / factor
                probs = probs / probs.sum(dim=1, keepdim=True)
            return probs

# Wrap our trained face model with defense
defended_model = DefendedFaceClassifier(face_model, noise_std=1.5, round_decimals=1)
defended_model.eval()

# Test clean accuracy under defense
with torch.no_grad():
    clean_preds = face_model(Xv).argmax(dim=1)
    defended_preds = defended_model(Xv).argmax(dim=1)
    
    clean_acc = (clean_preds == yv).float().mean().item() * 100
    defended_acc = (defended_preds == yv).float().mean().item() * 100
    
print(f"Model Utility Comparison:")
print(f"  Undefended Model Validation Accuracy: {clean_acc:.1f}%")
print(f"  Defended Model Validation Accuracy:   {defended_acc:.1f}%")
print(f"  Accuracy Drop (Utility Cost):          {clean_acc - defended_acc:.1f}%")


## **7. Comparative Reconstruction & Defense Evaluation** <a name="evaluation"></a>

We now execute the identical gradient-ascent inversion attack against the **Defended Model**. Because the defense injects stochastic perturbation into the logits, the gradient trajectories are scrambled, and the optimizer cannot successfully extract the biometric identity.


In [ ]:
print(f'Starting Model Inversion against DEFENDED model for {len(TARGET_SUBJECTS)} subjects...')
defended_inverted_faces = {}

for subject_id in TARGET_SUBJECTS:
    print(f'\n>>> Inverting Defended Subject ID: {subject_id}')
    defended_inverted_faces[subject_id] = invert_image(
        defended_model, subject_id, n_iter=2000, lr=0.01, tv_weight=1e-4, l2_weight=1e-3
    )

# Compute Mean Squared Error (MSE) against the true mean faces
mse_undefended = []
mse_defended = []

for subj in TARGET_SUBJECTS:
    true_mean = Xf_tr[yf_tr == subj].mean(axis=0).reshape(64, 64)
    mse_undef = np.mean((inverted_faces[subj] - true_mean) ** 2)
    mse_def = np.mean((defended_inverted_faces[subj] - true_mean) ** 2)
    mse_undefended.append(mse_undef)
    mse_defended.append(mse_def)

# Plot 3-row comparison: True Mean vs Undefended vs Defended
fig, axes = plt.subplots(3, len(TARGET_SUBJECTS), figsize=(3.5 * len(TARGET_SUBJECTS), 9))

for col, subj in enumerate(TARGET_SUBJECTS):
    true_mean_face = Xf_tr[yf_tr == subj].mean(axis=0).reshape(64, 64)
    
    # Row 1: True Training Mean Face
    axes[0, col].imshow(true_mean_face, cmap='gray')
    axes[0, col].set_title(f'Subject {subj}: Ground Truth', fontsize=11, fontweight='bold')
    axes[0, col].axis('off')
    
    # Row 2: Inverted Face (Undefended)
    axes[1, col].imshow(inverted_faces[subj], cmap='gray')
    axes[1, col].set_title(f'Undefended Inversion\nMSE: {mse_undefended[col]:.4f}', fontsize=10, color='crimson')
    axes[1, col].axis('off')
    
    # Row 3: Inverted Face (Defended)
    axes[2, col].imshow(defended_inverted_faces[subj], cmap='gray')
    axes[2, col].set_title(f'Defended Inversion\nMSE: {mse_defended[col]:.4f}', fontsize=10, color='darkgreen')
    axes[2, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Undefended (Vulnerable)', fontsize=12, fontweight='bold', color='crimson')
axes[2, 0].set_ylabel('Defended (Sanitized)', fontsize=12, fontweight='bold', color='darkgreen')

plt.suptitle('Biometric Model Inversion Attack vs. Privacy Defense', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("\n" + "=" * 65)
print("Reconstruction Fidelity Analysis:")
print(f"  Average Undefended Inversion MSE: {np.mean(mse_undefended):.4f}")
print(f"  Average Defended Inversion MSE:   {np.mean(mse_defended):.4f}")
print("  Defense successfully degrades biometric feature convergence.")
print("=" * 65)


## **8. Hands-on Exploratory Exercises** <a name="exercises"></a>

### **Exercise 1: Regularization Hyperparameter Tuning**
Analyze the effect of the Total Variation weight $\lambda_{\text{TV}}$ by running inversion for Subject `0` with $\lambda_{\text{TV}} \in \{0, 10^{-4}, 10^{-2}, 1.0\}$:
- When $\lambda_{\text{TV}} = 0$, what does the reconstructed image look like?
- When $\lambda_{\text{TV}} = 1.0$, what happens to fine biometric features (eyes, nose, mouth)?
- What is the optimal balance between confidence maximization and spatial smoothness?

### **Exercise 2: Impact of Model Depth on Inversion Fidelity**
Train a shallow 1-hidden-layer network ($4096 \to 64 \to 10$) and compare its inverted reconstructions with our 3-layer `FaceNet`. Does increasing network depth make reconstructions clearer or more abstract? Explain why.

### **Exercise 3: Defense Tuning — The Privacy-Utility Curve**
Evaluate `DefendedFaceClassifier` across a spectrum of noise parameters: $\sigma \in \{0.1, 0.5, 1.0, 2.0, 5.0\}$. For each value, record:
1. Validation accuracy on the face recognition task.
2. Inversion reconstruction MSE against the ground-truth mean face.
Plot the Pareto frontier of Validation Accuracy vs. Reconstruction MSE.

### **Exercise 4: Black-Box Inversion via Numerical Gradients**
Implement a black-box model inversion function where analytical backpropagation `loss.backward()` is disallowed. Estimate the gradient $\nabla_{\mathbf{x}} f_y(\mathbf{x})$ using Simultaneous Perturbation Stochastic Approximation (SPSA) or finite differences with $K=50$ random queries per iteration. How many queries are required to achieve recognizable face reconstruction?


## **9. Conclusion & Next Steps** <a name="conclusion"></a>

### **Core Takeaways**
1. **Models Memorize Class Archetypes:** Deep classifiers inadvertently encode recognizable representations of their training classes. White-box gradient ascent easily extracts these archetypes unless specific defenses are deployed.
2. **Natural Image Priors Are Essential:** Pure confidence maximization produces high-frequency adversarial noise. Inversion attacks require image priors (Total Variation and $L_2$ penalties) to constrain reconstructions to realistic biometric manifolds.
3. **Output Sanitization Mitigates Inversion:** Perturbing logits with calibrated noise (stochastic defense) and rounding output probabilities degrades the gradient landscape, preventing the optimizer from reconstructing recognizable faces while preserving utility on clear inputs.
4. **Formal Protection Requires Differential Privacy:** Empirical output perturbation helps against naive gradient ascent, but formal reconstruction bounds require training with **Differentially Private Stochastic Gradient Descent (DP-SGD)**, which guarantees bounded information leakage per sample (covered in Module 7).

### **MITRE ATLAS Mapping**
- **Tactic:** [AML.TA0005: Exfiltration](https://atlas.mitre.org/tactics/AML.TA0005)
- **Technique:** [AML.T0023: Model Inversion / Infer Training Data](https://atlas.mitre.org/techniques/AML.T0023)

### **Key References**
- Fredrikson, M., Jha, S., & Ristenpart, T. (2015). *Model Inversion Attacks that Exploit Confidence Information and Basic Countermeasures*. ACM CCS. [DOI:10.1145/2810103.2813677](https://doi.org/10.1145/2810103.2813677).
- Yang, Z., Zhang, J., Chen, E. C., & Huang, Z. (2019). *Neural Network Inversion in Adversarial Setting via Analyzing Predictions*. IJCAI.
- He, Z., Zhang, T., & Lee, R. B. (2019). *Model Inversion Attacks Against Collaborative Inference*. ACM ACSAC.
